# Niatnya mengguanakan BertSUm tapi ternyata gajadi karena 
### 1. Gagal terus memanggil BERTSum model dari hugging face
### 2. Karena target nya extractive summary dan ternyata memecah / split kalimat dari paragraf tidak bisa menemukan pola

## KESIMPULAN DARI SINI KARENA PENGATURAN SPLIT SULIT DILAKUKAN KARENA VARIATIF NYA KONTENT MAKA DIRASA UNTUK EXTRACTIVE SUMMARY SANGAT SULIT DILAKUKAN, SEHINGGA PENGGUNAAN BERTSUM TIDAK DILANJUTKAN

In [2]:
# Cek GPU
import torch
print(torch.cuda.is_available())

True


###***Ambil Dataset dari Kaggle***

In [3]:
# Upload ke Google Colab

from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"nalaprogroup","key":"6e69ae1b1b69e1a4d23d6edf0680d84f"}'}

In [4]:
# Set Permission
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [5]:
#Install Kaggle CLI
!pip install kaggle

In [6]:
# Download Dataset
# url : https://www.kaggle.com/datasets/nalaprogroup/clean-data-project2

!kaggle datasets download -d nalaprogroup/clean-data-project2

Dataset URL: https://www.kaggle.com/datasets/nalaprogroup/clean-data-project2
License(s): CC0-1.0
100% 15.7M/15.7M [00:00<00:00, 160MB/s]



In [7]:
# Extract File
!unzip clean-data-project2.zip

Archive:  clean-data-project2.zip
  inflating: dev_data_convert.json   
  inflating: dev_text_clean.json     
  inflating: test_data_convert.json  
  inflating: test_text_clean.json    
  inflating: train_data_convert.json  
  inflating: train_text_clean.json   


In [8]:
# memindahkan file ke sample_data/liputan6
!mkdir -p /content/sample_data/liputan6

!mv dev_data_convert.json /content/sample_data/liputan6
!mv train_data_convert.json /content/sample_data/liputan6
!mv test_data_convert.json /content/sample_data/liputan6

!mv dev_text_clean.json /content/sample_data/liputan6
!mv test_text_clean.json /content/sample_data/liputan6
!mv train_text_clean.json /content/sample_data/liputan6

###***Dataset***

Dataset - 01. Validasi Dataset

In [9]:
# Load 1 sample dulu
import json

with open('/content/sample_data/liputan6/train_text_clean.json') as f:
    data_train = json.load(f)

print("Jml train_text_clean :",len(data_train))

with open('/content/sample_data/liputan6/dev_text_clean.json') as g:
    data_dev = json.load(g)

print("Jml dev_text_clean :",len(data_dev))


with open('/content/sample_data/liputan6/test_text_clean.json') as h:
    data_test = json.load(h)

print("Jml test_text_clean :",len(data_test))


sample = data_train[0]

print("ID:\n", sample['id'])
print("ARTICLE:\n", sample['clean_article'])
print("\nSUMMARY:\n", sample['extractive_summary'])

Jml train_text_clean : 5000
Jml dev_text_clean : 5000
Jml test_text_clean : 5000
ID:
 198679
ARTICLE:
 Puluhan petani di Polewali Mandar, Sulawesi Barat, membakar lahan pertanian mereka. Ini dilakukan sebagai bentuk protes atas kinerja pemerintah yang dinilai lamban menangani bencana kekeringan akibat jebolnya bendungan Sekka-Sekka pasacabanjir bandang Januari lalu. Jebolnya bendungan membuat distribusi air terhenti. Menurut Abdul Rasyid, petani setempat, mereka kesal dan frustrasi karena padinya tak segera mendapat pasokan air yang cukup. Bahkan, sebagian petani menyerahkan lahannya pada peternak sapi untuk dijadikan pakan ternak. Kepala Dusun Nepo, Kecamatan Wonomulyo, Logawali, menjelaskan, bendungan sudah hampir dua bulan rusak. Namun hingga kini belum ada upaya perbaikan yang berarti. Pemerintah beralasan karena tidak ada dana perbaikan. Bencana kekeringan akibat kurangnya pasokan air merupakan yang kesekian kalinya. Padahal setiap kali musim tanam para petani harus mengeluarkan b

In [10]:
# Split jadi kalimat

import re

def split_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return sentences

sentences = split_sentences(sample['clean_article'])

for i, s in enumerate(sentences):
    print(i, s)

0 Puluhan petani di Polewali Mandar, Sulawesi Barat, membakar lahan pertanian mereka.
1 Ini dilakukan sebagai bentuk protes atas kinerja pemerintah yang dinilai lamban menangani bencana kekeringan akibat jebolnya bendungan Sekka-Sekka pasacabanjir bandang Januari lalu.
2 Jebolnya bendungan membuat distribusi air terhenti.
3 Menurut Abdul Rasyid, petani setempat, mereka kesal dan frustrasi karena padinya tak segera mendapat pasokan air yang cukup.
4 Bahkan, sebagian petani menyerahkan lahannya pada peternak sapi untuk dijadikan pakan ternak.
5 Kepala Dusun Nepo, Kecamatan Wonomulyo, Logawali, menjelaskan, bendungan sudah hampir dua bulan rusak.
6 Namun hingga kini belum ada upaya perbaikan yang berarti.
7 Pemerintah beralasan karena tidak ada dana perbaikan.
8 Bencana kekeringan akibat kurangnya pasokan air merupakan yang kesekian kalinya.
9 Padahal setiap kali musim tanam para petani harus mengeluarkan biaya hingga jutaan rupiah.
10 Bahkan tidak jarang berhutang kepada tengkulak hanya 

In [11]:
# Identifikasi tipe extractive_summary
print(type(sample['extractive_summary']))
print(sample['extractive_summary'])

<class 'list'>
[0, 3]


In [12]:
# Validasi otomatis  - Extractive Summary
def validate_data(sample):
    sentences = split_sentences(sample['clean_article'])
    summary = sample['extractive_summary']

    if isinstance(summary[0], int):
        # index-based
        for idx in summary:
            if idx >= len(sentences):
                return False
        return True
    else:
        # text-based
        for s in summary:
            if not any(s in sent for sent in sentences):
                return False
        return True

valid_count = 0

for i in range(5000):  # cek 100 sample dulu
    if validate_data(data_train[i]):
        valid_count += 1

print("Valid:", valid_count, "/ 5000")

Valid: 4961 / 5000


*Jika nilai ke-valid-an tidak memuaskan dibawah ini cara melihat data yang tidak valid*

In [13]:
# Ambil index data tidak valid - Extractive Summary
invalid_indices = []

for i, sample in enumerate(data_train):
    if not validate_data(sample):
        invalid_indices.append(i)

print("Jumlah tidak valid:", len(invalid_indices))
print("Contoh index:", invalid_indices[:10])

Jumlah tidak valid: 39
Contoh index: [100, 149, 559, 881, 982, 1016, 1036, 1081, 1224, 1316]


In [14]:
# Tampilkan isi data yang tidak valid - Extractive Summary

for i in invalid_indices[:5]:  # tampilkan 5 saja dulu
    print(f"\n=== DATA INDEX {i} ===")
    print("ARTICLE:\n", data_train[i]['clean_article'])
    print("\nEXTRACTIVE SUMMARY:\n", data_train[i]['extractive_summary'])


=== DATA INDEX 100 ===
ARTICLE:
 Klub Raksasa Liga Inggris, Manchester United ditaklukan oleh klub MLS, Kansas City Wizards dalam rangkaian tur Setan Merah di empat kota di Amerika Serikat. Di pertandingan yang mencetak rekor jumlah penonton sepakbola terbanyak di Stadion Kansas City ini, Davy Arnaud dan Kei Kamara mencetak gol-gol kemenangan bagi Wizards sementara satu satunya gol Setan Merah dicetak oleh penyerang Dimitar Berbatov. Meski bermain dengan sepuluh orang sejak menit ke-40, akan tetapi Wizards berhasil mencuri kemenangan dan membuat para suporter yang menyaksikan pertandingan dibawah terik matahari musim panas bangga kepada para pemain mereka. Tanpa pemain-pemain utama yang diistirahatkan usai perhelatan Piala Dunia kemarin, United masih tetap tampil dengan nama-nama beken seperti Ryan Giggs, Dimitar Berbatov, Nani, dam Paul Scholes. Dalam laga ini, Wizards membuka skor pada menit ke 11 lewat Davey Arnaud yang berhasil mengelabui jebakan offside Setan Merah. Manchester Un

In [15]:
# Validasi otomatis  - Abstractive Summary/Clean_Summary

empty_summary_idx = []

for i, sample in enumerate(data_train):
    summary = sample.get('clean_summary', None)

    if summary is None or summary.strip() == "":
        empty_summary_idx.append(i)

print("Jumlah summary kosong:", len(empty_summary_idx))
print("Contoh index:", empty_summary_idx[:10])

Jumlah summary kosong: 0
Contoh index: []


In [16]:
# Cek Clean_Summary yang kosong
for i in empty_summary_idx[:5]:
    print(f"\n=== DATA INDEX {i} ===")
    print("ARTICLE:\n", data_train[i]['clean_article'])
    print("\nSUMMARY:\n", data_train[i]['clean_summary'])

###***Preprocessing untuk BERTSum***

In [17]:
def protect_special_cases(text):
    # 1. Protect angka desimal (15.30)
    text = re.sub(r'(\d)\.(\d)', r'\1<dot>\2', text)

    # 2. Protect domain (TMZ.com)
    text = re.sub(r'\b([A-Za-z]+)\.(com|co|id|net|org)\b', r'\1<dot>\2', text)

    # 3. Protect inisial nama (M. Satrio)
    text = re.sub(r'\b([A-Z])\.\s+', r'\1<dot> ', text)

    # 4. Protect kata tertentu seperti "Dokter."
    text = re.sub(r'\b(Dokter|Dr|dr|Prof|Ir|H|Hj)\.\s+', r'\1<dot> ', text)

    return text

def fix_number_spacing(text):
    # gabungkan angka yang kepisah setelah restore
    text = re.sub(r'(\d)\.\s+(\d)', r'\1.\2', text)
    return text

# Split kalimat + buat label

def split_sentences(text):
    text = protect_special_cases(text)

    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)

    # restore kembali <dot> → .
    sentences = [s.replace('<dot>', '.') for s in sentences]

    # FIX tambahan
    sentences = [fix_number_spacing(s) for s in sentences]

    return sentences

def prepare_data(sample):
    sentences = split_sentences(sample['clean_article'])
    summary = sample['extractive_summary']

    labels = [0] * len(sentences)

    if isinstance(summary[0], int):
        for idx in summary:
            if idx < len(labels):
                labels[idx] = 1
    else:
        for i, sent in enumerate(sentences):
            for s in summary:
                if s.strip() == sent.strip():  # 🔥 FIX
                    labels[i] = 1

    return sentences, labels

In [58]:
# Test 1 sample

idx_data_sample=702
print("id :",data_train[idx_data_sample]["id"])
print("article :",data_train[idx_data_sample]["clean_article"])
print("summary :",data_train[idx_data_sample]["extractive_summary"])

print("id :",data_train[idx_data_sample]["id"])
sentences, labels = prepare_data(data_train[idx_data_sample])

for s, l in zip(sentences, labels):
    print(l, s)

id : 257464
article : Libur akhir tahun dimanfaatkan pesinetron Shireen Sungkar untuk mengisi kembali semangatnya. Penat dengan rutinitas, kekasih Adly Fairuz itu memboyong kakak, ibu, dan sepupunya, berlibur ke Bali selama beberapa hari. Gadis kelahiran Jakarta, 28 Januari 1992 itu pun dengan semangat menceritakan liburannya di Pulau Dewata. " Berhubung macet, aku naik ojek tiap hari. Naik mobil trus berhenti di tengah jalan. Macet banget, " kata Shireen seperti diberitakan Halo Selebriti di SCTV, Selasa (5/1). Pemeran utama dalam sinetron Cinta Fitri itu mengaku sangat menyukai pantai meski menderita alergi. " Kalau kena air laut gatel, tapi untung bawa salepnya, " kata Shireen sambil tersenyum. Meski bukan pertama kali ke Bali, personel The Sisters itu masih ingin kesana. " Masih banyak banget di Bali yang tempatnya bagus-bagus, dan aku belum ke sana, " kata putri pasangan Mark Sungkar dan Fanny Bauty tersebut. Soal rencana pada 2010, Shireen berharap bisa lebih baik dari tahun lalu

# KESIMPULAN DARI SINI KARENA PENGATURAN SPLIT SULIT DILAKUKAN KARENA VARIATIF NYA KONTENT MAKA DIRASA UNTUK EXTRACTIVE SUMMARY SANGAT SULIT DILAKUKAN, SEHINGGA PENGGUNAAN BERTSUM TIDAK DILANJUTKAN